In [17]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
from sklearn.metrics import confusion_matrix,classification_report

In [2]:
df=pd.read_csv( r"C:\Users\rosha\Desktop\ds\project roshaan\AD.csv")
df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


In [3]:
df.drop(["PatientID", "DoctorInCharge"], axis=1, inplace=True, errors='ignore')

In [4]:
numerical_cols = ["Age", "BMI", "AlcoholConsumption", "PhysicalActivity", "DietQuality", 
                      "SleepQuality", "SystolicBP", "DiastolicBP", "CholesterolTotal", 
                      "CholesterolLDL", "CholesterolHDL", "CholesterolTriglycerides",
                      "MMSE", "FunctionalAssessment", "ADL"]
    
categorical_cols = [c for c in df.columns if c not in numerical_cols and c != "Diagnosis"]
    

In [5]:
scaler = StandardScaler()
num_existing = [c for c in numerical_cols if c in df.columns]
df[num_existing] = scaler.fit_transform(df[num_existing])

In [6]:
categories = [sorted(df['Ethnicity'].unique().tolist()), sorted(df['EducationLevel'].unique().tolist())]
encoder = OrdinalEncoder(categories=categories)
df[['Ethnicity', 'EducationLevel']] = encoder.fit_transform(df[['Ethnicity', 'EducationLevel']])

In [7]:
x = df.drop("Diagnosis", axis=1)
y = df.Diagnosis
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.1, random_state=42)
    

In [8]:
cat_indices = [x.columns.get_loc(col) for col in categorical_cols if col in x.columns]
smote = SMOTENC(categorical_features=cat_indices, random_state=42)
x_res, y_res = smote.fit_resample(x_train, y_train)

In [9]:
modelbest_decision1=DecisionTreeClassifier(criterion="entropy",max_depth=5,max_features=None,min_samples_leaf=4,min_samples_split=2,random_state=42)

In [10]:
bagg_best=BaggingClassifier(estimator=modelbest_decision1,n_estimators=100,max_samples=0.7,max_features=1.0,bootstrap=True,bootstrap_features=False,random_state=42)
model=bagg_best.fit(x_res,y_res)

In [11]:
model.score(x_test,y_test)

0.958139534883721

In [15]:
y_pred=model.predict(x_test)

In [21]:
confusion_matrix(y_test,y_pred)

array([[138,   4],
       [  5,  68]])

In [20]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       142
           1       0.94      0.93      0.94        73

    accuracy                           0.96       215
   macro avg       0.95      0.95      0.95       215
weighted avg       0.96      0.96      0.96       215



In [12]:
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(x.columns.tolist(), "feature_columns.pkl")
joblib.dump(num_existing, "numerical_features.pkl")
joblib.dump(categorical_cols, "categorical_features.pkl")

print("✅ Model saved successfully!")

✅ Model saved successfully!
